# DCPBST: BRAC (Human Breast Cancer) Spatial Domain Identification

This notebook trains the DCPBST model on the BRAC (Human Breast Cancer) Visium dataset.

## Contents
1. Setup
2. Imports & Helper Functions
3. Load & Preprocess BRAC Data
4. Image Features
5. Model Training
6. Clustering & Metrics
7. Visualization
8. Save Results


In [ ]:
# ============================================================
# SETUP: Configure paths and imports for reproducibility
# ============================================================
import os
import sys

# Auto-detect repository root by walking up directories
_current = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_current, 'dcpbst_package')):
        break
    _current = os.path.dirname(_current)
else:
    _current = os.getcwd()

REPO_ROOT = _current
DATA_DIR = os.path.join(REPO_ROOT, 'data')
FIGURES_DIR = os.path.join(REPO_ROOT, 'figures')
RESULTS_DIR = os.path.join(REPO_ROOT, 'saved_results')
CONFIGS_DIR = os.path.join(REPO_ROOT, 'configs')

# Change to repo root so relative paths work
os.chdir(REPO_ROOT)

# Add dcpbst_package to import path
sys.path.insert(0, os.path.join(REPO_ROOT, 'dcpbst_package'))
print(f'Repo root: {REPO_ROOT}')
print(f'Data dir: {DATA_DIR}')

# Set random seeds for reproducibility
import numpy as np
import torch
import random
seed = 100
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = 'cuda'
else:
    device = 'cpu'
print(f'Device: {device}')


In [ ]:
import sys
import os
os.chdir(REPO_ROOT)
from dcpbst_package.model_827 import Dcpbst
from dcpbst_package.utils import *
from dcpbst_package.config_loader import load_config
from dcpbst_package.preprocess import *
from dcpbst_package.evals import *
from PIL import Image
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, \
    homogeneity_completeness_v_measure, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.preprocessing import LabelEncoder
import torch
import random
import json
import cv2
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm import tqdm
import scipy.sparse as sp
from scipy.sparse.csc import csc_matrix
from scipy.sparse.csr import csr_matrix
from scipy.spatial.distance import cdist
import ot

Image.MAX_IMAGE_PIXELS = None

# Reload config for BRAC
config = load_config('brac')
print('BRAC config loaded:', json.dumps(config, indent=2))

# Random seeds
seed = config.get('seed', 100)
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'CUDA available. GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('CUDA not available. Using CPU.')

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def load_visium_with_labels(path, label_col='fine_annot_type'):
    """Load Visium data and add Ground Truth labels from metadata.tsv."""
    adata = sc.read_visium(path, load_images=True)
    adata.var_names_make_unique()

    # Read metadata.tsv as labels
    metadata_path = os.path.join(path, "metadata.tsv")
    if os.path.exists(metadata_path):
        meta_df = pd.read_csv(metadata_path, sep="\t")
        assert label_col in meta_df.columns, f"{label_col} not in metadata.tsv"
        adata.obs["Ground Truth"] = meta_df[label_col].values
    else:
        raise FileNotFoundError("metadata.tsv not found")

    # Remove unlabeled entries
    adata = adata[~pd.isnull(adata.obs["Ground Truth"])]
    return adata


def preprocess_rna_dlpfc(adata):
    """Preprocess RNA: filter genes, HVG selection, normalize, log1p."""
    sc.pp.filter_genes(adata, min_counts=3)
    sc.pp.filter_cells(adata, min_counts=3)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata[:, adata.var['highly_variable']]


def extract_image_features_resnet(
        adata,
        data_path,
        backbone='ResNet50',
        device='cpu',
        img_save_dir='cut_img_BRCA',
        img_feat_name='Img_feat_brca.npy',
):
    """Extract image features using pretrained ResNet50."""
    if backbone == 'ResNet50':
        img_model = models.resnet50(pretrained=True)
        img_model.to(device)
        img_model.eval()

    # Create directory for cut images
    if not os.path.exists(img_save_dir):
        os.makedirs(img_save_dir)

    cut_save_path = os.path.join(img_save_dir, 'cut_img_BRCA.npy')
    feat_save_path = os.path.join(img_save_dir, img_feat_name)

    img_size = 224
    full_image = cv2.imread(os.path.join(data_path, "spatial", "full_image.tif"))
    full_image = cv2.cvtColor(full_image, cv2.COLOR_BGR2RGB)

    # Cut image patches if not already done
    if not os.path.exists(cut_save_path):
        print("Cutting image patches...")
        patches = []
        for x, y in adata.obsm['spatial']:
            x, y = int(x), int(y)
            patch = full_image[y - img_size:y + img_size, x - img_size:x + img_size]
            patches.append(patch)
        patches = np.array(patches)
        np.save(cut_save_path, patches)
        print(f"Saved cut images to {cut_save_path}")
    else:
        patches = np.load(cut_save_path)

    # Extract features if not already done
    if not os.path.exists(feat_save_path):
        print("Extracting image features with ResNet50...")
        spot_img = patches.astype(np.float32) / 255.0
        tensor = torch.from_numpy(spot_img)
        img_feat = []
        for i, element in tqdm(enumerate(tensor), total=len(tensor), desc='Extracting features'):
            element = element.permute(2, 0, 1).unsqueeze(0)  # [1, 3, 224, 224]
            element = element.to(device)
            with torch.no_grad():
                ret = img_model(element)
            ret = ret.data.cpu().numpy().ravel()
            img_feat.append(ret)
        img_feat = np.array(img_feat)
        np.save(feat_save_path, img_feat)
        print(f"Saved image features to {feat_save_path}")
    else:
        img_feat = np.load(feat_save_path)
        print(f"Loaded pre-computed image features from {feat_save_path}")

    adata.obsm['feat_img'] = img_feat
    return adata


def purity_score(y_true, y_pred):
    """Compute purity score."""
    cm = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(cm, axis=0)) / np.sum(cm)


def refine_label(adata, radius=50, key='label'):
    """Refine cluster labels by spatial neighbor majority voting."""
    n_neigh = radius
    new_type = []
    old_type = adata.obs[key].values

    # Calculate spatial distance
    position = adata.obsm['spatial']
    distance = ot.dist(position, position, metric='euclidean')

    n_cell = distance.shape[0]

    for i in range(n_cell):
        vec = distance[i, :]
        index = vec.argsort()
        neigh_type = []
        for j in range(1, n_neigh + 1):
            neigh_type.append(old_type[index[j]])
        max_type = max(neigh_type, key=neigh_type.count)
        new_type.append(max_type)

    new_type = [str(i) for i in list(new_type)]
    return new_type


def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=2024):
    """Clustering using the mclust algorithm (R package)."""
    np.random.seed(random_seed)
    import rpy2.robjects as robjects
    robjects.r.library("mclust")

    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
    r_random_seed = robjects.r['set.seed']
    r_random_seed(random_seed)
    rmclust = robjects.r['Mclust']

    res = rmclust(rpy2.robjects.numpy2ri.numpy2rpy(adata.obsm[used_obsm]), num_cluster, modelNames)
    mclust_res = np.array(res[-2])

    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int')
    adata.obs['mclust'] = adata.obs['mclust'].astype('category')
    return adata


def calculate_clustering_matrix(df, y_pred, ground_truth):
    """Compute clustering evaluation metrics and append to dataframe."""
    ari = adjusted_rand_score(y_pred, ground_truth)
    nmi = normalized_mutual_info_score(y_pred, ground_truth)
    purity = purity_score(y_pred, ground_truth)
    homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_pred, ground_truth)
    print("ARI={}, NMI={}, Purity={}, V_Measure={}".format(ari, nmi, purity, v_measure))

    new_row = pd.DataFrame([{
        'ARI': ari,
        'NMI': nmi,
        'Purity': purity,
        'Homogeneity': homogeneity,
        'Completeness': completeness,
        'V_Measure': v_measure
    }])

    df = pd.concat([df, new_row], ignore_index=True)
    return df


def clustering(adata, n_clusters=7, radius=50, key='emb', method='mclust',
               start=0.1, end=3.0, increment=0.01, refinement=False):
    """Spatial clustering based on the learned representation."""
    # PCA dimensionality reduction before clustering
    pca = PCA(n_components=20, random_state=2024)
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding

    if method == 'mclust':
        adata = mclust_R(adata, used_obsm='emb_pca', num_cluster=n_clusters)
        adata.obs['domain'] = adata.obs['mclust']
    elif method == 'kmeans':
        kmeans = KMeans(n_clusters=n_clusters).fit(embedding)
        kmeans_result = [i + 1 for i in kmeans.labels_]
        adata.obs['domain'] = list(map(lambda x: str(x), kmeans_result))
    elif method == 'leiden':
        res = search_res(adata, n_clusters, use_rep='emb_pca', method=method,
                         start=start, end=end, increment=increment)
        sc.tl.leiden(adata, random_state=0, resolution=res)
        adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
        res = search_res(adata, n_clusters, use_rep='emb_pca', method=method,
                         start=start, end=end, increment=increment)
        sc.tl.louvain(adata, random_state=0, resolution=res)
        adata.obs['domain'] = adata.obs['louvain']

    if refinement:
        new_type = refine_label(adata, radius, key='domain')
        adata.obs['domain'] = new_type


def plot_on_histology(clusters, locs, im, scale, s=10, title=None):
    """Plot clustering results on histology image."""
    from matplotlib import colors as mcolors
    locs = locs * scale
    locs = locs.round().astype('int')
    im_cropped = im[(locs['4'].min() - 10):(locs['4'].max() + 10),
                    (locs['5'].min() - 10):(locs['5'].max() + 10)]
    locs = locs - locs.min() + 10
    cmap1 = mcolors.ListedColormap(
        [cmap_tab70(np.array(i)) for i in range(len(np.unique(clusters)))])
    plt.imshow(im_cropped, alpha=0.7)
    plot = plt.scatter(x=locs['5'], y=locs['4'], c=clusters, cmap=cmap1, s=s)
    if title:
        plt.title(title)
    plt.axis('off')
    return plot


In [ ]:
# ============================================================
# Step 1: Load & Preprocess BRAC Data
# ============================================================
data_path = os.path.join(DATA_DIR, 'Human_breast')
print(f"Loading BRAC data from: {data_path}")

# Load data with Ground Truth labels
adata = load_visium_with_labels(data_path, label_col='fine_annot_type')
print(f"Loaded data shape: {adata.shape}")
print(f"Ground Truth categories: {sorted(adata.obs['Ground Truth'].unique())}")

# Preprocess: filter genes, HVG selection, normalize, log1p
adata = preprocess_rna_dlpfc(adata)
print(f"After preprocessing: {adata.shape}")

# Extract RNA feature matrix
if scipy.sparse.issparse(adata.X):
    scrna = adata.X.A
else:
    scrna = adata.X
print(f"scRNA feature shape: {scrna.shape}")

# Number of clusters from Ground Truth
n_clusters = len(set(adata.obs['Ground Truth']))
print(f"Number of clusters (from Ground Truth): {n_clusters}")


In [ ]:
# ============================================================
# Step 2: Image Features
# ============================================================
img_feat_path = os.path.join(DATA_DIR, 'cut_img_BRCA', 'Img_feat_brca.npy')
print(f"Looking for pre-computed image features at: {img_feat_path}")

if os.path.exists(img_feat_path):
    print("Loading pre-computed ResNet features...")
    img_feat = np.load(img_feat_path)
    adata.obsm['feat_img'] = img_feat
    print(f"Loaded image features shape: {img_feat.shape}")
else:
    print("Pre-computed features not found. Extracting via ResNet...")
    adata = extract_image_features_resnet(
        adata,
        data_path=data_path,
        backbone='ResNet50',
        device=device,
        img_save_dir=os.path.join(DATA_DIR, 'cut_img_BRCA'),
        img_feat_name='Img_feat_brca.npy',
    )
    img_feat = adata.obsm['feat_img']
    print(f"Extracted image features shape: {img_feat.shape}")


In [ ]:
# ============================================================
# Step 3: Model Training
# ============================================================
print("Initializing DCPBST model with BRAC config...")
print(f"  latent_dim: {config.get('latent_dim', 1024)}")
print(f"  pca_n_components: {config.get('pca_n_components', 1000)}")
print(f"  embed_dim: {config.get('embed_dim', 512)}")
print(f"  num_heads: {config.get('num_heads', 8)}")
print(f"  output_dim: {config.get('output_dim', 256)}")
print(f"  neighbors: {config.get('neighbors', 7)}")
print(f"  gat_dropout: {config.get('gat_dropout', 0.5)}")
print(f"  n_clusters: {n_clusters}")

# Initialize model
model = Dcpbst(
    [scrna, img_feat],
    sparse=False,
    device=device,
    n_clusters=n_clusters,
    adata=adata,
    neighbors=config.get('neighbors', 7),
    latent_dim=config.get('latent_dim', 1024),
    final_feature=config.get('output_dim', 256),
)

# Train the model
print(f"\nTraining for {config.get('epochs', 900)} epochs...")
embedding = model.fit(
    epochs=config.get('epochs', 900),
    lr=config.get('lr', 1e-3),
    w_cls=config.get('w_cls', 10.0),
    w_recon=config.get('w_recon', 10.0),
    w_kl=config.get('w_kl', 0.1),
    w_pro=config.get('w_pro', 2.0),
    w_info=config.get('w_info', 3.0),
    w_dgi=config.get('w_dgi', 0.1),
    w_clu=config.get('w_clu', 1.0),
    diagnose_every_n_epochs=config.get('diagnose_every_n_epochs', 20),
)

print(f"Training complete. Embedding shape: {embedding.shape}")

# Store embeddings in adata
adata.obsm['emb'] = embedding
adata.obsm['emb1'] = embedding

# Compute PCA on embeddings (50 components for visualization)
print("Computing PCA on embeddings (50 components)...")
pca_50 = PCA(n_components=50, random_state=2024)
embedding_pca = pca_50.fit_transform(embedding)
adata.obsm['emb1_pca'] = embedding_pca
print(f"PCA embedding shape: {embedding_pca.shape}")


In [ ]:
# ============================================================
# Step 4: Clustering & Metrics
# ============================================================
print("Running mclust clustering...")
print(f"  n_clusters: {n_clusters}")
print(f"  model: EEE")

# Clustering with spatial refinement
radius = config.get('refine_radius', 50)
clustering(adata, n_clusters=n_clusters, radius=radius,
           key='emb', method='mclust', refinement=True)

print(f"\nClustering complete.")
print(f"Cluster distribution:\n{adata.obs['domain'].value_counts().sort_index()}")

# Calculate metrics
print("\nComputing clustering metrics...")
eval_df = pd.DataFrame(columns=['ARI', 'NMI', 'Purity', 'Homogeneity', 'Completeness', 'V_Measure'])
eval_df = calculate_clustering_matrix(eval_df, adata.obs['domain'], adata.obs['Ground Truth'])
print(f"\nClustering metrics:\n{eval_df}")

# Store final clusters
final_clusters = adata.obs['domain']
unique_clusters = final_clusters.unique()
num_final_clusters = len(unique_clusters)
print(f"\nFinal clusters: {sorted(unique_clusters)}")
print(f"Number of final clusters: {num_final_clusters}")


In [ ]:
# ============================================================
# Step 5: Visualization
# ============================================================

print("Building neighbor graph on embedding 'emb1_pca' for UMAP...")
sc.pp.neighbors(adata, use_rep='emb1_pca', n_neighbors=10)
print("Computing UMAP...")
sc.tl.umap(adata)

# UMAP plot with cluster labels
plt.rcParams['figure.figsize'] = (8, 6)
sc.pl.umap(adata, color='domain', title='DCPBST Clustering (BRAC)', frameon=False)

# UMAP plot with Ground Truth
if 'Ground Truth' in adata.obs.columns:
    sc.pl.umap(adata, color='Ground Truth', title='Ground Truth', frameon=False)

# Spatial plot with cluster labels
try:
    ari_score = eval_df['ARI'].iloc[0]
except:
    ari_score = 0.0

plt.rcParams['figure.figsize'] = (6, 6)
sc.pl.spatial(
    adata,
    color='domain',
    title=f'DCPBST Clustering (ARI={ari_score:.2f})',
)

# Spatial plot with Ground Truth
if 'Ground Truth' in adata.obs.columns:
    sc.pl.spatial(
        adata,
        color='Ground Truth',
        title='Ground Truth',
    )

# Histology overlay
print("\nGenerating histology overlay plot...")
locs_path = os.path.join(data_path, "spatial", "tissue_positions_list.csv")
locs = pd.read_csv(locs_path, header=None, index_col=0)
locs.columns = ['in_tissue', 'array_row', 'array_col', '4', '5']
locs = locs.loc[adata.obs_names]

clusters_int = adata.obs['domain'].astype(int).values
im = plt.imread(os.path.join(data_path, "spatial", "tissue_hires_image.png"))
with open(os.path.join(data_path, "spatial", "scalefactors_json.json"), 'r') as f:
    scalefactors = json.load(f)
scale = scalefactors['tissue_hires_scalef']

plt.rcParams['figure.figsize'] = (8, 8)
plot_on_histology(clusters_int, locs, im, scale, s=10,
                  title=f'BRAC Histology Overlay (ARI={ari_score:.2f})')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Step 6: Save Results
# ============================================================

# Create results directory if needed
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save AnnData with cluster annotations
h5ad_path = os.path.join(RESULTS_DIR, 'dcpbst_adata_with_clusters.h5ad')
adata.write(h5ad_path)
print(f"Saved AnnData to: {h5ad_path}")

# Save model state_dict
model_path = os.path.join(RESULTS_DIR, 'dcpbst_model.pth')
torch.save(model.state_dict(), model_path)
print(f"Saved model to: {model_path}")

# Save metrics CSV
metrics_path = os.path.join(RESULTS_DIR, 'dcpbst_clustering_metrics.csv')
eval_df.to_csv(metrics_path, index=False)
print(f"Saved metrics to: {metrics_path}")

print("\n" + "=" * 60)
print("BRAC DCPBST pipeline completed successfully!")
print(f"  Results dir: {RESULTS_DIR}")
print(f"  AnnData: {h5ad_path}")
print(f"  Model: {model_path}")
print(f"  Metrics: {metrics_path}")
print(f"  ARI: {ari_score:.4f}")
print("=" * 60)
